# Normalization & adjusted EBITDA

Walk from **reported EBITDA** to **adjusted EBITDA** using `NormalizationConfig` with fixed add-backs, **percentage-of-revenue** fees, and **capped** synergies.

## Concept

`NormalizationConfig` is JSON-serializable: each adjustment has a `type` of `fixed` (per-period map), `percentage_of_node`, or `formula`. Caps reference a **base node** and a **percentage limit**. Python exposes `normalize` (JSON) and `normalize_to_dicts` (row dicts).

In [ ]:
from finstack.statements import ModelBuilder, Evaluator, FinancialModelSpec
from finstack.statements_analytics import run_sensitivity, evaluate_scenario_set, goal_seek, trace_dependencies, explain_formula, run_variance, evaluate_dcf, run_corporate_analysis, pl_summary_report, credit_assessment_report
import json
print("Loaded finstack.statements + finstack.statements_analytics")

from finstack.statements import NormalizationConfig, normalize, normalize_to_dicts

PERIODS = ["2025Q1", "2025Q2"]


def z(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("norm-demo")
b.periods("2025Q1..Q2", None)
b.value("revenue", z([1000.0, 1100.0]))
b.value("ebitda", z([120.0, 135.0]))
spec = b.build()
res = Evaluator().evaluate(spec)

cfg = NormalizationConfig("ebitda")
print("Empty config:", cfg, "| adjustments:", cfg.adjustment_count)


In [ ]:
print("Normalization JSON config + engine")
cfg_json = {
    "target_node": "ebitda",
    "adjustments": [
        {
            "id": "restructuring",
            "name": "Restructuring add-back",
            "value": {
                "type": "fixed",
                "amounts": {"2025Q1": 8.0, "2025Q2": 5.0},
            },
        },
        {
            "id": "mgmt_fee",
            "name": "Mgmt fee (% of revenue)",
            "value": {"type": "percentage_of_node", "node_id": "revenue", "percentage": 0.02},
        },
        {
            "id": "synergies",
            "name": "Synergies (capped at 15% of EBITDA)",
            "value": {
                "type": "fixed",
                "amounts": {"2025Q1": 30.0, "2025Q2": 25.0},
            },
            "cap": {"base_node": "ebitda", "value": 0.15},
        },
    ],
}
full_cfg = NormalizationConfig.from_json(json.dumps(cfg_json))
print("Loaded adjustments:", full_cfg.adjustment_count)

norm_json = normalize(res, full_cfg)
print("normalize JSON:", norm_json)

rows = normalize_to_dicts(res, full_cfg)
for row in rows:
    print("period", row["period"], "base", row["base_value"], "final", row["final_value"], "steps", len(row["adjustments"]))


## Takeaways

- Keep a **machine-readable adjustment policy** (`from_json`) next to human memos for audit.
- **Percentage fees** and **caps** prevent runaway add-backs while staying transparent in `normalize_to_dicts`.
- Merge normalized series back into models with downstream nodes if you need **forecast adjusted EBITDA**.